#### Control Flow

Control flow means : Controlling how Execution moves through the graph depending on the situation.

instead of saying: Step by step , next this will excecute Node B. 
``` markdown
we say if this happens --> Node B
Else --> Node C
Retry if Failed
Loop until success
Stop if user interrupts 
"" This Intelligence is called Control Flow""
```


1. Conditional Excecution

Control which node runs next based on the current state. In LangGraph this is done with conditional edges — a routing function reads state and returns the name of the next node.

Example : Think of it like a traffic signal. The same intersection (state) can route cars (execution) in different directions depending on the light condition.

![alt text](image-13.png)



In [4]:
from langgraph.graph import StateGraph, END, START
from typing import TypedDict

# Step 1: Define State
class State(TypedDict):
    score: int
    result: str   # separate field for the label

# Step 2: Define nodes
def grade_high(state):
    return {"result": f"{state['score']} → PASS"}

def grade_low(state):
    return {"result": f"{state['score']} → FAIL"}

# Step 3: Router function — reads state, returns node name
def route_by_score(state):
    if state["score"] >= 50:
        return "grade_high"
    return "grade_low"

# Step 4: Wire it up
builder = StateGraph(State)
builder.add_node("grade_high", grade_high)
builder.add_node("grade_low", grade_low)

# Entry point is START, conditional edges route to the correct node
builder.add_conditional_edges(
    START,                  # source: the very beginning
    route_by_score,         # routing function
    {"grade_high": "grade_high", "grade_low": "grade_low"}
)

# Each grading node goes to END
builder.add_edge("grade_high", END)
builder.add_edge("grade_low", END)

# Step 5: Compile and invoke
graph = builder.compile()

# Run it and print the result
result = graph.invoke({"score": 75, "result": ""})
print(result)
# Output: {'score': 75, 'result': '75 → PASS'}

result2 = graph.invoke({"score": 30, "result": ""})
print(result2)
# Output: {'score': 30, 'result': '30 → FAIL'}


{'score': 75, 'result': '75 → PASS'}
{'score': 30, 'result': '30 → FAIL'}


Branching

Branching is conditional execution with multiple destinations. One node's output can fan out to 2, 3, or more paths simultaneously (parallel branching) or one-at-a-time (exclusive branching).

Exclusive Branch (One path)
Router picks exactly one next node. Like an if/elif/else chain — only one branch executes.

![alt text](image-14.png)


Parallel branch (all paths) : Multiple nodes start simultaneously. Results are merged later. Like Promise.all().



![alt text](image-15.png)




In [ ]:
from typing import Literal
# Exclusive branchin - router returns one node name.

def classify_intent(state)->Literal["billing","support","sales"]:
    intent = state["intent"]
    if intent == "payment":
        return "billing"
    if intent == "help":
        return "support"
    return "sales"

builder.add_conditional_edges("intake",classify_intent)

# Parallel Branching - add multiple edges from the same source
builder.add_edge("fetch_data","db_query") # all three
builder.add_edge("fetch_data","api_call") # start at
builder.add_edge("fetch_data","cache") # the same time

# they all converge at merge node.
builder.add_edge("db_query", "merge_results")
builder.add_edge("api_call", "merge_results")
builder.add_edge("cache",    "merge_results")

Loops & Cycles

A loop in langgraph is an edge that sends excecution back to a previous node. Combined with a conditional edge, this creates agent-style "think-->act-->check-->repeat" cycles.

Like a QA reviewer who sends a document back for revision until it passes. The loop keeps running until the exit condition is met.

![alt text](image-16.png)

In [4]:
# Classic ReAct-style loop: agent → tools → agent → ... → END
from langgraph.graph import StateGraph, END
from typing import TypedDict,Literal

class AgentState(TypedDict):
    messages:list[str]
    iteraton: int # this is to track loop count
    done:bool

llm = ""
def agent(state):
    #llm decides what to do next
    response = llm.invoke(state["messages"])
    return {
        "messages" : state["messages"] + [response],
        "iteration": state["iteration"] + 1
    }
execute_tool = " "
def run_tools(state):
    "excecute whatver tool the agent requested"
    result = execute_tool(state["messages"][-1])
    return {"messages": state["messages"] + [result]}

# ⚠ IMPORTANT: Always add a loop guard!
def should_continue(state) -> Literal["run_tools", "__end__"]:
    last = state["messages"][-1]
    if state["iteration"] >= 10:      # safety limit
        return END
    if is_final_answer(last):           # LLM is done
        return END
    return "run_tools"                 # keep looping

builder = StateGraph(AgentState)
builder.add_node("agent", agent)
builder.add_node("run_tools", run_tools)
builder.set_entry_point("agent")

builder.add_conditional_edges("agent", should_continue)
builder.add_edge("run_tools", "agent")   # the loop-back edge

#⚡ Always add a loop guard — a max iteration count or a "final answer" detector — to prevent infinite loops. LangGraph does not stop them automatically.


Retry Logic

when a node fails (API timeout,LLM refusal,bad output), we want to retry it before giving up. Langgraph supports retry through nodel-level retry policies and manual loop-based retry patterns.

1. Built-in retry policy
Pass retry=RetryPolicy(...) to add_node(). Automatic exponential backoff. Best for transient failures (network, rate limits).

2. Manual retry loop (flixible)
Track attempt in state. Route back to the same node on failure. Full control over what "success" means.


In [ ]:
# Option A: Built-in RetryPolicy (LangGraph ≥0.1)
from langgraph.pregel import RetryPolicy

builder.add_node(
    "call_api",
    call_api_node,
    retry = RetryPolicy(
        max_attempts = 3,
        initial_interval = 1.0, # second before first retry
        backoff_factor = 2.0 # doubles each time: 1s, 2s, 4s
        jitter = True # random spread to avoid thundering herd
        retry_on = (TimeoutError,RateLimitError)
    )
)
# LangGraph handles the retry automatically — no extra code needed


In [ ]:
# Option B: Manual retry loop (more control)
class State(TypedDict):
    query:str
    result:str
    attempts:str
    error:str

def call_llm(state):
    try:
        result = llm.invoke(state["query"])
        return {"result":result,"error":None}
    except Exception as e:
        return {"error": str(e), "attempts": state["attempts"] + 1}

def retry_router(state) -> Literal["call_llm", "handle_failure", "__end__"]:
    if state["error"] is None:
        return END                           # success!
    if state["attempts"] < 3:
        return "call_llm"                   # retry
    return "handle_failure"                 # give up gracefully

builder.add_node("call_llm", call_llm)
builder.add_node("handle_failure", handle_failure)
builder.add_conditional_edges("call_llm", retry_router)

Error Handling

Errors in LangGraph can be handled at three levels: inside a node (try/except), at the graph level (error edges), or via fallback nodes that recover gracefully.

Think of error handling like an airport: small issues (gate change) are handled locally; major issues (weather) trigger a redirect; and catastrophic failures (cancellation) route passengers to a fallback destination.



In [8]:
# # Level 1: Handle errors inside the node itself

external_api = ""
def risky_node(state):
    try:
        result = external_api(state["input"])
        return {"result": result, "status": "ok"}
    except TimeoutError:
        return {"result": None, "status": "timeout"}
    except ValueError as e:
        return {"result": None, "status": "bad_input", "error_msg": str(e)}

In [ ]:
# Level 2: Route to a fallback node based on status
def error_router(state) -> Literal["success_node", "fallback_node", "log_error"]:
    status = state.get("status")
    if status == "ok":       return "success_node"
    if status == "timeout":  return "fallback_node"   # use cached data
    return "log_error"                               # unrecoverable

def fallback_node(state):
    # Graceful degradation: return cached/default result
    cached = get_from_cache(state["input"])
    return {"result": cached, "source": "cache"}

def log_error(state):
    # Write to error log, notify monitoring, end gracefully
    log_to_datadog(state["error_msg"])
    return {"result": "Error: unable to process request"}

builder.add_conditional_edges("risky_node", error_router)


In [ ]:
# Level 3: Wrap the entire graph in try/except at run time
graph = builder.compile()

try:
    result = graph.invoke({"input": "process this"})
except GraphInterruptException:
    print("Graph was interrupted by a breakpoint")
except Exception as e:
    print(f"Unhandled graph error: {e}")

Interruptions (Human-in-the-Loop)

LangGraph can pause excecution mid-graph and wait for human input before continuing. This is essential for approval flows, content review, and any scenario where AI must not proceed without human sign-off.

Like a surgeon who must verbally confirm "correct patient, correct site" before making the first cut — the workflow physically stops until a human says "proceed".

 ![alt text](image-17.png)

In [ ]:
# Step 1: Compile graph WITH interrupt_before
graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["execute_plan"]   # pause BEFORE this node
)

# Step 2: Run until the interrupt
thread = {"configurable": {"thread_id": "approval-1"}}
graph.invoke({"plan": "delete all records"}, config=thread)
# ↑ Graph pauses here. State is saved to checkpointer.

# Step 3: Human reviews the plan (your UI logic)
state = graph.get_state(thread)
print("Plan to approve:", state.values["plan"])

# Step 4a: Approve — resume from where it stopped
graph.invoke(None, config=thread)   # None = no new input, just continue

# Step 4b: Reject — update state before resuming
graph.update_state(thread, {"approved": False, "reason": "too risky"})
graph.invoke(None, config=thread)   # resumes, now conditional edge routes to cancel

In [ ]:
# interrupt_after — pause AFTER a node completes (review its output)
graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_after=["draft_email"]   # stop after draft is ready
)

# You can also use interrupt() inside a node for dynamic pausing
from langgraph.types import interrupt

def review_node(state):
    human_input = interrupt({          # pauses here, returns human's reply
        "question": "Approve this plan?",
        "plan": state["plan"]
    })
    return {"approved": human_input == "yes"}

🔑 Interruptions require a checkpointer (like MemorySaver or SqliteSaver) so state is preserved while waiting for human input. Without it, the graph can't be resumed.
